# 01 · Pipeline check with pretrained SmolVLA models (Step A3)

| model | expected | what it proves |
|---|---|---|
| `smolvla_base` | ~0% success, arm moves | plumbing works end to end |
| `smolvla_libero` (`HuggingFaceVLA/smolvla_libero`) | high success | our image flip, state, keys, normalization and gripper sign are right |

Run `00_colab_setup.ipynb` once first (it downloads the demos). A100 recommended; a T4 works but is slower.
Weights are downloaded to `MyDrive/groundedvla/models/` the first time, then copied to local disk each session.

In [ ]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

In [ ]:
# Fast local copies of weights already on Drive (first run: run_sim downloads them to Drive)
for folder in ["smolvla_base", "smolvla_libero"]:
    if (Path(paths["models"]) / folder / "config.json").exists():
        stage_to_local(f"models/{folder}")

## Rollouts (10 per task, videos saved to Drive)

In [ ]:
EPISODES = 10
for model in ["smolvla_base", "smolvla_libero"]:
    for task in ["T1", "T2", "T3"]:
        !python scripts/run_sim.py {model} --task {task} --episodes {EPISODES} --viewer none --save-video

Optional: compare execution modes on `smolvla_libero` (naive N=1/10 vs temporal ensemble).

In [ ]:
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --horizon 1
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --mode temporal_ensemble

In [ ]:
from pathlib import Path
from IPython.display import Video, display
for v in sorted(Path(paths["outputs"]).glob("run_sim/smolvla_libero/videos/*.mp4"))[:3]:
    print(v.name); display(Video(str(v), embed=True, width=512))

In [ ]:
import json
import pandas as pd
rows = []
for f in Path(paths["outputs"], "run_sim").glob("*/*.jsonl"):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
df.groupby(["variant", "task"]).agg(episodes=("success", "size"), success=("success", "mean"),
                                    steps=("steps", "mean"), policy_call_ms=("latency_ms_median", "median"))